In [1]:
import operator
from googleapiclient.discovery import build
from dotenv import load_dotenv
import os
import re

In [2]:
load_dotenv(override=True)
API_KEY = os.getenv("YOUTUBE_API_KEY")

In [3]:

youtube = build("youtube", "v3", developerKey=API_KEY)
video_ids = ['Gvt4TwGpqOs',
 'Jj1-zb38Yfw',
 'Jj1-zb38Yfw',
 'tr5Fapv80Cw',
 'MrD9tCNpOvU',
 '-pqzyvRp3Tc']


In [4]:
request = youtube.videos().list(
            part="snippet,statistics,contentDetails",
            id=",".join(video_ids)
            )

response = request.execute()
print(response)

{'kind': 'youtube#videoListResponse', 'etag': '9x9w1wqP5xVkI-SSNaVqFqnAJOI', 'items': [{'kind': 'youtube#video', 'etag': '5_yef3kcme6kS76aNjI_Tq0aZhU', 'id': 'Gvt4TwGpqOs', 'snippet': {'publishedAt': '2026-02-15T14:00:24Z', 'channelId': 'UCbiGcwDWZjz05njNPrJU7jA', 'title': 'Explaining Agentic AI: The Good, the Bad & the Ugly', 'description': 'What are agentic AI and agentic operating systems? And what are their potential benefits, drawbacks and longer-term implications?\n\nFor further information, the websites and articles included in this video are as follows:\n\nPROTOCOLS:\n\nMCP website: https://modelcontextprotocol.io/docs/getting-started/intro\n\nGoogle announces the A2A protocol: https://developers.googleblog.com/en/a2a-a-new-era-of-agent-interoperability/\n\nIBM’s ACP integrated into A2A: https://lfaidata.foundation/communityblog/2025/08/29/acp-joins-forces-with-a2a-under-the-linux-foundations-lf-ai-data/\n\nA2A Protocol website: https://a2a-protocol.org/latest/\n\n\nORCHESTRATI

In [12]:
def get_counts(video_ids):
    request = youtube.videos().list(
            part="snippet,statistics,contentDetails",
            id=",".join(video_ids)
            )

    response = request.execute()
    print(response["items"])
    video_summaries = {}
    for video in response["items"]:
        views = int(video["statistics"]["viewCount"])
        likes = int(video["statistics"]["likeCount"])
        comments = int(video["statistics"]["commentCount"])
        video_summaries[video["id"]] = {"views" : views, "likes" : likes, "comments" : comments, "title": video["snippet"]["title"], "channelTitle" : video["snippet"]["channelTitle"]}
    return video_summaries



In [13]:
video_ids_dict = get_counts(video_ids)
video_ids_dict

[{'kind': 'youtube#video', 'etag': 'LxhXFoAcFtbR44UMAZo_8o8_3W8', 'id': 'Gvt4TwGpqOs', 'snippet': {'publishedAt': '2026-02-15T14:00:24Z', 'channelId': 'UCbiGcwDWZjz05njNPrJU7jA', 'title': 'Explaining Agentic AI: The Good, the Bad & the Ugly', 'description': 'What are agentic AI and agentic operating systems? And what are their potential benefits, drawbacks and longer-term implications?\n\nFor further information, the websites and articles included in this video are as follows:\n\nPROTOCOLS:\n\nMCP website: https://modelcontextprotocol.io/docs/getting-started/intro\n\nGoogle announces the A2A protocol: https://developers.googleblog.com/en/a2a-a-new-era-of-agent-interoperability/\n\nIBM’s ACP integrated into A2A: https://lfaidata.foundation/communityblog/2025/08/29/acp-joins-forces-with-a2a-under-the-linux-foundations-lf-ai-data/\n\nA2A Protocol website: https://a2a-protocol.org/latest/\n\n\nORCHESTRATION PLATFORMS:\n\nMicrosoft AutoGen: https://www.microsoft.com/en-us/research/project/a

{'Gvt4TwGpqOs': {'views': 82829,
  'likes': 8553,
  'comments': 1022,
  'title': 'Explaining Agentic AI: The Good, the Bad & the Ugly',
  'channelTitle': 'ExplainingComputers'},
 'Jj1-zb38Yfw': {'views': 136913,
  'likes': 3286,
  'comments': 66,
  'title': 'Agentic AI Explained So Anyone Can Get It!',
  'channelTitle': 'ByteMonk'},
 'tr5Fapv80Cw': {'views': 62640,
  'likes': 1797,
  'comments': 42,
  'title': 'Building Agentic AI Workloads – Crash Course',
  'channelTitle': 'freeCodeCamp.org'},
 'MrD9tCNpOvU': {'views': 59982,
  'likes': 853,
  'comments': 22,
  'title': 'Agentic AI Design Patterns Introduction and walkthrough | Amazon Web Services',
  'channelTitle': 'Amazon Web Services'},
 '-pqzyvRp3Tc': {'views': 199414,
  'likes': 2450,
  'comments': 57,
  'title': 'What is Agentic AI? An Easy Explanation For Everyone',
  'channelTitle': 'Bernard Marr'}}

In [7]:
def get_authority_scores(video_ids_dict):
    max_views= max(v['views'] for v in video_ids_dict.values())
    min_views= min(v['views'] for v in video_ids_dict.values())
    max_likes= max(v['likes'] for v in video_ids_dict.values())
    min_likes= min(v['likes'] for v in video_ids_dict.values())
    max_comments= max(v['comments'] for v in video_ids_dict.values())
    min_comments= min(v['comments'] for v in video_ids_dict.values())
    authority_scores_dict = video_ids_dict.copy()
    for video_id in video_ids_dict.keys():
        current_views = video_ids_dict[video_id]['views']
        current_likes = video_ids_dict[video_id]['likes']
        current_comments = video_ids_dict[video_id]['comments']
        normalized_views = (current_views - min_views) / (max_views - min_views)
        normalized_likes = (current_likes - min_likes) / (max_likes - min_likes)
        normalized_comments = (current_comments - min_comments) / (max_comments - min_comments)
        authority_score = 0.5 * normalized_views + 0.3 * normalized_likes + 0.2 * normalized_comments
        authority_scores_dict[video_id]["authority_score"] = round(authority_score,2)
    return authority_scores_dict



In [8]:

authority_scores_dict = get_authority_scores(video_ids_dict)
print(authority_scores_dict)





{'Gvt4TwGpqOs': {'views': 82828, 'likes': 8553, 'comments': 1022, 'title': 'Explaining Agentic AI: The Good, the Bad & the Ugly', 'channelTitle': 'ExplainingComputers', 'authority_score': 0.58}, 'Jj1-zb38Yfw': {'views': 136911, 'likes': 3286, 'comments': 66, 'title': 'Agentic AI Explained So Anyone Can Get It!', 'channelTitle': 'ByteMonk', 'authority_score': 0.38}, 'tr5Fapv80Cw': {'views': 62639, 'likes': 1797, 'comments': 42, 'title': 'Building Agentic AI Workloads – Crash Course', 'channelTitle': 'freeCodeCamp.org', 'authority_score': 0.05}, 'MrD9tCNpOvU': {'views': 59982, 'likes': 853, 'comments': 22, 'title': 'Agentic AI Design Patterns Introduction and walkthrough | Amazon Web Services', 'channelTitle': 'Amazon Web Services', 'authority_score': 0.0}, '-pqzyvRp3Tc': {'views': 199412, 'likes': 2450, 'comments': 57, 'title': 'What is Agentic AI? An Easy Explanation For Everyone', 'channelTitle': 'Bernard Marr', 'authority_score': 0.57}}


In [9]:
def get_video_summaries(authority_scores_dict):
    video_summaries = []
    for video_id in authority_scores_dict.keys():
        score = authority_scores_dict[video_id]["authority_score"]
        if score >= 0.8:
            recommendation = "recommended as core curriculum material"
        elif score >= 0.6:
            recommendation = "recommended as primary supporting material"
        elif score >= 0.4:
            recommendation = "suitable as supplementary learning"
        else:
            recommendation = "optional reference material"
        text = f"""
        The video "{video_ids_dict[video_id]['title']}" published by
        {video_ids_dict[video_id]['channelTitle']}
        achieves an authority score of {score:.2f}.

        Based on engagement and quality metrics, it is {recommendation}
        for learners exploring this topic.
        """
        video_summaries.append(text)
    return video_summaries



In [10]:
print([re.sub(r"\n|\s{2,}", " ", video_summary).strip() for video_summary in get_video_summaries(authority_scores_dict)])

['The video "Explaining Agentic AI: The Good, the Bad & the Ugly" published by  ExplainingComputers  achieves an authority score of 0.58.   Based on engagement and quality metrics, it is suitable as supplementary learning  for learners exploring this topic.', 'The video "Agentic AI Explained So Anyone Can Get It!" published by  ByteMonk  achieves an authority score of 0.38.   Based on engagement and quality metrics, it is optional reference material  for learners exploring this topic.', 'The video "Building Agentic AI Workloads – Crash Course" published by  freeCodeCamp.org  achieves an authority score of 0.05.   Based on engagement and quality metrics, it is optional reference material  for learners exploring this topic.', 'The video "Agentic AI Design Patterns Introduction and walkthrough | Amazon Web Services" published by  Amazon Web Services  achieves an authority score of 0.00.   Based on engagement and quality metrics, it is optional reference material  for learners exploring th